# FoodLens：提示词复述诊断（不训练）

复用现有代码 SHA/CUDA/ZIP/模型准备流程。检查 16 张 train 的全部监督 token，再在同一组 2 train + 2 val 图片上比较 A/B/C 零样本提示；不读取 test 作实验，不增加 epoch/rank，不启动 100 步或完整训练。所有输出来自 Colab 实际执行，Notebook 不预填成绩。


In [ ]:
import os, sys, json, subprocess, platform, uuid
from pathlib import Path
from datetime import datetime, timezone

if not Path("/content").is_dir() or "google.colab" not in sys.modules and not os.environ.get("COLAB_RELEASE_TAG"):
    try:
        import google.colab
    except ImportError as exc:
        raise RuntimeError("请在 Google Colab 托管 GPU 运行时运行；不支持本地训练") from exc
import torch
assert torch.cuda.is_available(), "请先选择 NVIDIA GPU 运行时；这里不会退回 CPU 训练"
print(platform.python_version(), torch.__version__, torch.cuda.get_device_name(0))


## 获取成员 C 分支代码
先在推送后用 `git rev-parse HEAD` 取得最终完整 40 位 SHA，填写下方 EXPECTED_CODE_COMMIT（不能填旧的 6a8ac50）。留空、缩写或实际分支 HEAD 不一致都会停止；不会自动 checkout 其他版本或 main。已有目录只读检查，不自动拉取、切换、覆盖或清理。若目录版本旧，请在新运行时重试，或自行处理目录。所有命令均检查退出状态。


In [ ]:
BRANCH = "feat/member-c-diet-record"
REPO_URL = "https://github.com/liuxy2006/FoodLens.git"
REPO = Path("/content/FoodLens-member-c")
EXPECTED_CODE_COMMIT = ""  # 必填：推送本次最终提交后，粘贴 git rev-parse HEAD 的完整 40 位 SHA
import re
if not re.fullmatch(r"[0-9a-fA-F]{40}", EXPECTED_CODE_COMMIT):
    raise ValueError("请填写审核并推送后的最终完整 40 位 commit SHA；不要填写旧提交或短 SHA")
EXPECTED_CODE_COMMIT = EXPECTED_CODE_COMMIT.lower()
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
assert subprocess.check_output(["git", "-C", str(REPO), "branch", "--show-current"], text=True).strip() == BRANCH
assert subprocess.check_output(["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip() == REPO_URL
assert not subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip(), "代码目录有修改，先人工检查"
for relative in ["task2_diet_record/checkpoint.py", "task2_diet_record/integration.py", "colab/member_c_train.ipynb", "task2_diet_record/prepare_colab_data.py", "task2_diet_record/colab_dependencies.py", "task2_diet_record/schema_experiment.py", "task2_diet_record/overfit_config.json", "task2_diet_record/prompt_diagnostics.py", "task2_diet_record/supervision_audit.py"]:
    assert (REPO / relative).is_file(), "远程分支尚无第二阶段代码；请审核后自行推送该分支，再开新运行时"
CODE_COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert CODE_COMMIT == EXPECTED_CODE_COMMIT, "代码版本与预期不一致"
os.chdir(REPO)
print("branch:", BRANCH, "commit:", CODE_COMMIT)
def run_module(module, *arguments):
    subprocess.run([sys.executable, "-m", module, *map(str, arguments)], cwd=REPO, check=True)


## 安装依赖
保留 Colab 预装的 CUDA PyTorch：安装前后在新进程读取 torch.__version__、torch.version.cuda、torch.cuda.is_available()；用精确约束锁定已安装的 torch/torchvision/torchaudio/triton，并先审查 pip dry-run 清单。若计划替换 GPU 运行时、安装 Windows/XPU 依赖，或安装后 CUDA 不可用，就停止。固定现有框架依赖，记录实际环境。若安装后提示需要重启，请重启会话后从头执行；已满足的依赖不会重复安装。


In [ ]:
# 仅在 Colab 执行：安装模块将保存 before/after、约束和 pip 计划。
INSTALL_REPORT = Path("/content/foodlens_install_" + uuid.uuid4().hex[:8])
run_module("task2_diet_record.colab_dependencies", "--output", INSTALL_REPORT)


## 挂载 Drive、设置路径和记录环境
数据只需以两个 ZIP 上传到 Drive；下一格自动识别 ZIP 层级、复制并解压到 Colab 本地。已有本地完整数据校验通过后直接复用；不完整/不一致数据会明确报错，不自动覆盖或清理。输出目录使用唯一运行 ID，断线后的恢复必须指定旧断点路径，不能覆盖旧实验。


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/FoodLens")
DATASET_DIR = DRIVE_ROOT / "dataset"
DATA_ROOT = Path("/content/foodlens_data")
ARCHIVE_CACHE = Path("/content/foodlens_archives")
REVISION = "7e3e67edbbed1bf9888184d9df282b700a323964"
MODEL_DIR = DRIVE_ROOT / "models" / ("SmolVLM-256M-Instruct-" + REVISION[:8])
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
RUN_ROOT = DRIVE_ROOT / "outputs" / ("prompt_diagnostics_" + RUN_ID)
RUN_ROOT.mkdir(parents=True, exist_ok=False)
os.environ["HF_HOME"] = str(DRIVE_ROOT / "cache" / "huggingface")
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import shutil
shutil.copytree(INSTALL_REPORT, RUN_ROOT / "dependency_install")
(RUN_ROOT / "source.json").write_text(json.dumps({"branch": BRANCH, "commit": CODE_COMMIT, "model_revision": REVISION}, indent=2))
(RUN_ROOT / "pip_freeze.txt").write_text(subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True))
(RUN_ROOT / "nvidia_smi.txt").write_text(subprocess.check_output(["nvidia-smi"], text=True))
run_module("task2_diet_record.device_probe", "--output", RUN_ROOT / "device_probe.json")
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
print("所有结果保存至:", RUN_ROOT)


In [ ]:
from task2_diet_record.prepare_colab_data import prepare_data
preparation = prepare_data(DATASET_DIR, DATA_ROOT, ARCHIVE_CACHE)
assert Path(preparation["data_root"]) == DATA_ROOT
print(json.dumps(preparation, ensure_ascii=False, indent=2))
(RUN_ROOT / "data_preparation.json").write_text(json.dumps(preparation, indent=2))
# 已核对全部 train/val/test 数量与图片引用；再对训练/验证图片做少量实际解码。
from task2_diet_record.data import DietDataset, read_annotations
for split in ("train", "val"):
    dataset = DietDataset(DATA_ROOT / "raw/annotations" / (split + ".jsonl"), DATA_ROOT, split)
    for index in range(min(8, len(dataset))):
        dataset[index]["image"].close()
    print(split, "前 8 张图片加载成功")
# test 在这里仅作静态完整性核对，标签不用于训练、选模、调参或提示词优化。
val_rows = read_annotations(DATA_ROOT / "raw/annotations/val.jsonl")[:2]
VAL_SMOKE = RUN_ROOT / "val_smoke.jsonl"
VAL_SMOKE.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in val_rows), encoding="utf-8")


## 核实并下载唯一指定模型
已查询官方仓库：必要文件约 516.6 MB，其中 safetensors 约 513.0 MB。先输出清单，再下载相同 revision 的模型与 Processor；复跑时缓存可复用。下载后核对大小和 LFS SHA256，不下载 Qwen/ONNX 等其他权重。


In [ ]:
run_module("task2_diet_record.prepare_model", "--revision", REVISION, "--destination", MODEL_DIR,
           "--manifest", RUN_ROOT / "model_manifest.json")
run_module("task2_diet_record.prepare_model", "--revision", REVISION, "--destination", MODEL_DIR,
           "--manifest", RUN_ROOT / "model_manifest.json", "--download")


## 最小实验：16 图监督审计 + 12 次零样本生成
A 是当前原始长提示，B 是短 JSON 提示，C 是短提示加人工格式示例（不是图片真值）。同一原始预训练模型、同一组 4 张图片、同一 512 token 上限和贪心解码，分别保存实际提示词/哈希。没有训练，也不会拿 B/C 去加载 A 的 Adapter。
可选填写已有完整 schema_step25 目录，只读验证后额外对同一 4 张图做 **A 提示词的训练后诊断**，与零样本结果分目录保存；这额外增加 4 次生成，不修改旧实验。


In [ ]:
EXISTING_STAGE25_DIR = ""
DIAGNOSTICS = RUN_ROOT / "probe"
arguments = ["--model-path", MODEL_DIR, "--revision", REVISION, "--data-root", DATA_ROOT,
             "--output", DIAGNOSTICS, "--max-new-tokens", 512]
if EXISTING_STAGE25_DIR:
    arguments += ["--existing-stage25-dir", EXISTING_STAGE25_DIR]
run_module("task2_diet_record.prompt_diagnostics", *arguments)


## 先看监督报告，再比较生成 token
supervision_audit_16.json 包含全部 16 条记录，逐条检查 JSON、前缀、标签、图像 token、截断和终止 token；单个样本失败也会保存报告并继续检查其他样本。
A/B/C 每条样本均保存完整 input/output/new token IDs、带特殊 token 的解码、输入尾部、前缀一致性和 EOS。若 new token 中仍有提示词片段且输入前缀校验通过，复述是模型真实生成，不能通过删除字符串掩盖。
零样本只有 4 张固定图，不能据此宣称泛化提升；C 若照抄格式示例的值，即使 Schema 合法也不能视为正确营养预测。


In [ ]:
audit = json.loads((DIAGNOSTICS / "supervision_audit_16.json").read_text())
print("监督检查:", audit["passed_samples"], "/", audit["samples"])
for sample in audit["samples_report"]:
    print(sample["dish_id"], sample["all_checks_passed"], sample.get("checks"), sample.get("error"))
for variant in ("A", "B", "C"):
    report = json.loads((DIAGNOSTICS / "zero_shot" / (variant + ".json")).read_text())
    print("PROMPT", variant, report["prompt"], report["prompt_sha256"], report["summary"])
    for prediction in report["predictions"]:
        trace = prediction.get("trace") or {}
        print(prediction["dish_id"], prediction["prediction"], prediction.get("error"))
        print("Input tail:", trace.get("input_tail_with_special_tokens"))
        print("New tokens only:", trace.get("new_output_with_special_tokens"))
        print("Prefix matches:", trace.get("input_prefix_matches"), "Echo:", prediction.get("echo"))
print("完整原始 token 报告:", DIAGNOSTICS)


## 暂停训练，先解释结果
此 Notebook 没有任何训练入口。若输入前缀不一致、16 条审计未全通过、终止 token 与生成停止配置不匹配，先定位这些问题，不继续到 100 步。
若前缀和监督都正常，A 在新 token 中复述而 B/C 明显减少，只支持提示复杂度相关的假设。需要训练时，按各提示词新建独立实验和签名，禁止复用 A 的 checkpoint 来声称 B/C 微调成功。
只有同一训练提示下格式合法率与复述情况改善、没有伪修复/模板问题、逐条监督通过，才重新审查是否继续原 A 实验至 100 步；不要仅凭 loss 下降作决定。
